# Figure 4: OBSALL (left) + AQUA (right)

This notebook generates the OBSALL and AQUA panels and combines them into a single publication-ready PDF.

In [1]:
from pathlib import Path
from string import ascii_lowercase
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import pandas as pd
import xarray as xr
from cartopy import crs as ccrs, feature as cfeature

from aqua.core.graphics import plot_single_map_diff, ConfigStyle
from aqua.core.logger import log_configure

/opt/homebrew/Caskroom/miniconda/base/envs/aqua-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [78]:
# Base paths
root = Path('/Users/mnurisso/src/DestinE_paper/Figure4')
obsall_dir = root / 'OBSALL'
aqua_dir = root / 'AQUA'
output_dir = root / 'figures' / 'combined'
output_dir.mkdir(parents=True, exist_ok=True)

# OBSALL inputs
station_list_path = obsall_dir / 'SYNOP' / 'synop_station_list.txt'
model_results_path = obsall_dir / 'mean-map-model_results.pickle'

# AQUA inputs
netcdf_dir = aqua_dir / 'netcdf'
icon_file_mask = str(netcdf_dir / '*ICON*.nc')
ifs_nemo_file_mask = str(netcdf_dir / '*IFS-NEMO*.nc')
ifs_fesom_file_mask = str(netcdf_dir / '*IFS-FESOM*.nc')
era5_file_mask = str(netcdf_dir / '*ERA5*.nc')

# Output file
combined_pdf = output_dir / 'fig4_obsall_aqua_combined.pdf'
combined_png = output_dir / 'fig4_obsall_aqua_combined.png'

In [81]:
def plot_combined_figure(output_path: Path, diff_range: float = 7.0, diff_levels: int = 16):
    import pickle

    # Read the stations list
    station_list = (
        pd.read_csv(station_list_path, sep=r"\s+")
        .rename(
            {
                "station@hdr_integer": "id",
                "longitude@hdr:real": "longitude",
                "latitude@hdr:real": "latitude",
                "elevation@hdr:real": "elevation",
            },
            axis="columns",
        )[["id", "longitude", "latitude", "elevation"]]
        .set_index("id")
    )

    # Load the OBSALL pre analysed data
    with open(model_results_path, "rb") as f:
        model_results = pickle.load(f)

    diff_min = 0.0
    diff_max = 0.0
    for station_stats in model_results.values():
        diff_min = min(
            diff_min,
            (station_stats["ann_mean_sim"] - station_stats["ann_mean_obs"]).min(),
        )
        diff_max = max(
            diff_max,
            (station_stats["ann_mean_sim"] - station_stats["ann_mean_obs"]).max(),
        )

    diff_extend = "both"
    obs_models = {
        "icon_hist_o25-1": "ICON",
        "ifs-nemo_hist_o25-1": "IFS-NEMO",
        "ifs-fesom_hist_o25-1": "IFS-FESOM",
    }
    obs_combined_models = {
        "icon_hist_o25-1": ["icon_hist_o25-1"],
        "ifs-nemo_hist_o25-1": ["ifs-nemo_hist_o25-1"],
        "ifs-fesom_hist_o25-1": ["ifs-fesom_hist_o25-1"],
    }

    # Set a common AQUA style for homogeneity in the plot
    loglevel = "WARNING"
    logger = log_configure(loglevel, "plot_biases")
    ConfigStyle(loglevel=loglevel)

    # Open AQUA data
    icon_climatology = xr.open_mfdataset(icon_file_mask)
    ifs_nemo_climatology = xr.open_mfdataset(ifs_nemo_file_mask)
    ifs_fesom_climatology = xr.open_mfdataset(ifs_fesom_file_mask)
    era5_climatology = xr.open_mfdataset(era5_file_mask)

    # Set AQUA details for plot
    aqua_models = [icon_climatology, ifs_nemo_climatology, ifs_fesom_climatology]
    aqua_titles = ["ICON", "IFS-NEMO", "IFS-FESOM"]
    aqua_panels = ["d)", "e)", "f)"]
    aqua_titles = [f"{aqua_panels[i]} {aqua_titles[i]}" for i in range(len(aqua_titles))]
    var = "u"

    def plot_station_map(ax, station_stats: list[pd.DataFrame]):
        """OBSALL plot routine to plot the single station in the final plot"""
        se_factor = 2 * np.sqrt(1 + 1 / len(station_stats))

        ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
        ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)
        ax.set_global()

        levels = (
            [-diff_range]
            + list(
                np.linspace(
                    -diff_range / 2, diff_range / 2, diff_levels + (1 - diff_levels % 2) - 2
                )
            )
            + [diff_range]
        )
        cmap = plt.get_cmap("RdBu_r", len(levels) + 1)
        norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend=diff_extend)

        mean_ann_mean_sim = sum(s["ann_mean_sim"] for s in station_stats) / len(station_stats)

        sig_mask = (
            mean_ann_mean_sim
            < (station_stats[0]["ann_mean_obs"] - station_stats[0]["ann_se_obs"] * se_factor)
        ) | (
            mean_ann_mean_sim
            > (station_stats[0]["ann_mean_obs"] + station_stats[0]["ann_se_obs"] * se_factor)
        )

        ax.scatter(
            station_list.loc[mean_ann_mean_sim.index][~sig_mask]["longitude"],
            station_list.loc[mean_ann_mean_sim.index][~sig_mask]["latitude"],
            c=mean_ann_mean_sim.loc[~sig_mask] - station_stats[0].loc[~sig_mask]["ann_mean_obs"],
            cmap=cmap,
            norm=norm,
            s=30,
            marker="^",
            transform=ccrs.PlateCarree(),
            edgecolors="grey",
            lw=0.5,
        )
        sc = ax.scatter(
            station_list.loc[mean_ann_mean_sim.index][sig_mask]["longitude"],
            station_list.loc[mean_ann_mean_sim.index][sig_mask]["latitude"],
            c=mean_ann_mean_sim.loc[sig_mask] - station_stats[0].loc[sig_mask]["ann_mean_obs"],
            cmap=cmap,
            norm=norm,
            s=30,
            marker="o",
            transform=ccrs.PlateCarree(),
            edgecolors="grey",
            lw=0.5,
        )

        legend_elements = [
            mpl.lines.Line2D([0], [0], marker="o", color="black", linestyle="", markersize=5, label="significant"),
            mpl.lines.Line2D([0], [0], marker="^", color="black", linestyle="", markersize=5, label="not significant"),
        ]
        ax.legend(handles=legend_elements, loc="lower left")

        return sc

    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(4, 2, height_ratios=[1, 1, 1, 0.12], wspace=0.02, hspace=0.25)

    for i, (model, model_name) in enumerate(obs_models.items()):
        a = ascii_lowercase[i]
        ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.Robinson())

        mean_ann_bias = np.mean(
            [
                np.mean(model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"])
                for m in obs_combined_models[model]
            ]
        )
        mean_ann_abs_bias = np.mean(
            [
                np.mean(np.abs(model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"]))
                for m in obs_combined_models[model]
            ]
        )
        ann_bias_positive = np.mean(
            [
                np.mean((model_results[m]["ann_mean_sim"] - model_results[m]["ann_mean_obs"]) > 0)
                for m in obs_combined_models[model]
            ]
        )

        ax1.set_title(
            f"{a}) {model_name} ({np.round(mean_ann_bias, 2)}, {np.round(mean_ann_abs_bias, 2)}, {int(np.round(ann_bias_positive * 100))})",
            loc="center", font='Arial', fontweight='bold', fontsize=16
        )

        sc_obs = plot_station_map(ax1, [model_results[m] for m in obs_combined_models[model]])

    cax_left = fig.add_subplot(gs[3, 0])
    cax_left.set_axis_off()
    cbar_left_ax = inset_axes(
        cax_left,
        width="80%",
        height="100%",
        loc="center",
        borderpad=0,
    )
    levels_obs = [-7.0, -3.5, -3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 7.0]
    cmap_obs = plt.get_cmap("RdBu_r", len(levels_obs) + 1)
    norm_obs = mcolors.BoundaryNorm(boundaries=levels_obs, ncolors=cmap_obs.N, extend=diff_extend)

    cbar_left = fig.colorbar(
        sc_obs,
        cax=cbar_left_ax,
        cmap=cmap_obs,
        norm=norm_obs,
        extend=diff_extend,
        boundaries=levels_obs,
        ticks=levels_obs,
        orientation="horizontal",
        label="2m temperature bias ($^{o}C$)",\
    )
    cbar_left.ax.tick_params(labelsize=12)
    cbar_left.ax.xaxis.label.set_fontsize(12)
    for boundary in levels_obs:
        cbar_left.ax.vlines(boundary, *cbar_left.ax.get_ylim(), color="black", linewidth=0.7)

    levels_aqua = np.arange(-7.5, 7.5 + 1, 1)
    cmap_aqua = "PuOr_r"
    for i, model in enumerate(aqua_models, 1):
        data = model[var]
        fig, ax = plot_single_map_diff(
            data=data,
            data_ref=era5_climatology[var],
            fig=fig,
            norm=None,
            ax_pos=(4, 2, i * 2),
            cmap=cmap_aqua,
            vmin_fill=-7.5,
            vmax_fill=7.5,
            vmin_contour=-10.0,
            vmax_contour=15.0,
            return_fig=True,
            cbar=False,
            sym=False,
            line_levels=6,
            nlevels=15,
            title_size=14,
            loglevel=loglevel,
        )

        ax.set_title(aqua_titles[i - 1], loc="center",
                     font='Arial', fontweight='bold', fontsize=16)

    cax_right = fig.add_subplot(gs[3, 1])
    cax_right.set_axis_off()
    cbar_right_ax = inset_axes(
        cax_right,
        width="80%",
        height="100%",
        loc="center",
        borderpad=0,
    )
    mappable = ax.collections[0]
    cbar_right = fig.colorbar(mappable, cax=cbar_right_ax, orientation="horizontal", label="Zonal wind speed bias (m/s)")
    cbar_right.set_ticks(levels_aqua)
    cbar_right.ax.set_xticklabels([f"{tick:.0f}" if tick == int(tick) else f"{tick:.1f}" for tick in levels_aqua])
    cbar_right.ax.tick_params(labelsize=12)
    cbar_right.ax.xaxis.label.set_fontsize(12)
    for boundary in levels_aqua:
        cbar_right.ax.vlines(boundary, *cbar_right.ax.get_ylim(), color="black", linewidth=0.7)

    fig.subplots_adjust(left=0.03, right=0.97, top=0.99, bottom=0.06, wspace=0.02, hspace=0.25)
    fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)    

In [76]:
plot_combined_figure(combined_pdf)

In [82]:
plot_combined_figure(combined_png)